In [5]:
# Cell 1: Install dependencies quietly
!pip install ucimlrepo joblib -q

# Cell 2: Imports
import os
import shutil
import joblib
import pandas as pd
import numpy as np
from google.colab import files
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

# 1. Fetch Dataset (UCI ID: 350)
print("Fetching dataset...")
credit_data = fetch_ucirepo(id=350)
X = credit_data.data.features.copy()
y = credit_data.data.targets.copy().iloc[:, 0].astype(int)

# 2. Stratified sampling (10,000 instances for clean memory execution)
X_sample, _, y_sample, _ = train_test_split(
    X, y, train_size=10000, stratify=y, random_state=42
)

# 3. 80/20 Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sample, y_sample, test_size=0.20, stratify=y_sample, random_state=42
)

# 4. Export test_data.csv with actual target column
test_export = X_test.copy()
test_export["default_payment_next_month"] = y_test.values
test_export.to_csv("test_data.csv", index=False)

# 5. Define pipelines
os.makedirs("model", exist_ok=True)

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "decision_tree": Pipeline([
        ("clf", DecisionTreeClassifier(max_depth=6, random_state=42))
    ]),
    "k_nearest_neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=9))
    ]),
    "gaussian_naive_bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GaussianNB())
    ]),
    "support_vector_machine": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(probability=True, kernel="rbf", C=1.0, random_state=42))
    ])
}

# 6. Fit, serialize, compute metrics, and print the table
results = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, f"model/{name}.pkl")

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name.replace("_", " ").title(),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_prob), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4)
    })

# Displays the table directly in the notebook output
print("\n=== MODEL COMPARISON TABLE ===")
print(pd.DataFrame(results).to_markdown(index=False))

# 7. Zip model folder and trigger downloads
shutil.make_archive("model", "zip", "model")

print("\nTriggering file downloads to your local machine...")
files.download("test_data.csv")
files.download("model.zip")

Fetching dataset...

=== MODEL COMPARISON TABLE ===
| Model                  |   Accuracy |    AUC |   Precision |   Recall |     F1 |    MCC |
|:-----------------------|-----------:|-------:|------------:|---------:|-------:|-------:|
| Logistic Regression    |     0.8075 | 0.7417 |      0.7209 |   0.2104 | 0.3257 | 0.3164 |
| Decision Tree          |     0.8235 | 0.7239 |      0.6894 |   0.3665 | 0.4786 | 0.4119 |
| K Nearest Neighbors    |     0.807  | 0.706  |      0.6443 |   0.2828 | 0.3931 | 0.3344 |
| Gaussian Naive Bayes   |     0.776  | 0.7322 |      0.4941 |   0.5679 | 0.5284 | 0.3841 |
| Support Vector Machine |     0.818  | 0.7315 |      0.699  |   0.31   | 0.4295 | 0.3797 |

Triggering file downloads to your local machine...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>